# 05 - Bronze: restaurantes (OpenStreetMap / Overpass)

Busca os estabelecimentos de comida do município de São Paulo e salva o JSON **bruto** no Volume.

- **Filtro:** `amenity` = restaurant, fast_food, cafe ou ice_cream
- **Área:** caixa (bounding box) do município de São Paulo
- **Resiliência:** até 3 tentativas, com espera entre elas, caso o Overpass responda com erro

**Saída:** `/Volumes/workspace/bronze/raw_files/restaurants/ingestion_date=AAAA-MM-DD/`

A última célula é um diagnóstico de qualidade: quantos registros têm `cuisine` e `opening_hours`. Restaurantes mudam pouco, então este notebook roda semanalmente.
```

In [0]:
import requests, json, os, time
from datetime import datetime
from zoneinfo import ZoneInfo

BASE_PATH = "/Volumes/workspace/bronze/raw_files/restaurants"
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Bounding box aproximada do município de São Paulo (sul, oeste, norte, leste)
BBOX = (-23.80, -46.83, -23.35, -46.36)
CIDADE = "sao_paulo"

agora = datetime.now(ZoneInfo("America/Sao_Paulo"))
ingestion_date = agora.strftime("%Y-%m-%d")
ts = agora.strftime("%H%M%S")

s, w, n, e = BBOX
query = f"""
[out:json][timeout:180];
nwr["amenity"~"^(restaurant|fast_food|cafe|ice_cream)$"]({s},{w},{n},{e});
out center tags;
"""

headers = {"User-Agent": "weather-food-recommendation/0.1 (projeto de portfolio)"}

In [0]:
payload = None
for tentativa in range(1, 4):
    r = requests.post(OVERPASS_URL, data={"data": query}, headers=headers, timeout=240)
    print(f"Tentativa {tentativa} -> status {r.status_code}")
    if r.status_code == 200:
        payload = r.json()
        break
    time.sleep(30 * tentativa)

if payload is None:
    raise RuntimeError("Overpass não respondeu com sucesso após 3 tentativas")

print("Elementos retornados:", len(payload["elements"]))

In [0]:
pasta = f"{BASE_PATH}/ingestion_date={ingestion_date}"
os.makedirs(pasta, exist_ok=True)
caminho = f"{pasta}/{CIDADE}_{ts}.json"

with open(caminho, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False)

print("Salvo em:", caminho)
display(dbutils.fs.ls(pasta))

In [0]:
from collections import Counter

amenities = Counter(el.get("tags", {}).get("amenity") for el in payload["elements"])
print(amenities)

total = len(payload["elements"])
com_cozinha = sum(1 for el in payload["elements"] if "cuisine" in el.get("tags", {}))
com_horario = sum(1 for el in payload["elements"] if "opening_hours" in el.get("tags", {}))
print(f"Com cuisine: {com_cozinha}/{total} | Com opening_hours: {com_horario}/{total}")